# Chapter 7: Gaussian Processes

A Gaussian Process (GP) places a prior over **functions**, not just over scalars.
This is essential in astrophysics for modelling correlated noise (stellar activity,
systematics) and for non-parametric signal reconstruction.

$$f(x) \sim \mathcal{GP}(m(x),\; k(x, x'))$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.linalg import cholesky, solve_triangular

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})
np.random.seed(42)
print("Ready.")


## 7.1 Kernels — The Language of Smoothness

The kernel k(x, x') encodes how correlated f(x) and f(x') are.
Different kernels encode different beliefs about the function.


In [ ]:
# Visualise different kernels and sample functions from each GP prior
def k_se(x, xp, sigma_f=1.0, ell=1.0):
    # Squared Exponential (RBF) kernel.
    return sigma_f**2 * np.exp(-0.5 * ((x - xp)/ell)**2)

def k_matern32(x, xp, sigma_f=1.0, ell=1.0):
    # Matérn 3/2 kernel.
    r = np.abs(x - xp) / ell
    return sigma_f**2 * (1 + np.sqrt(3)*r) * np.exp(-np.sqrt(3)*r)

def k_periodic(x, xp, sigma_f=1.0, ell=1.0, period=5.0):
    # Periodic kernel.
    return sigma_f**2 * np.exp(-2 * np.sin(np.pi*np.abs(x-xp)/period)**2 / ell**2)

def k_qp(x, xp, sigma_f=1.0, ell=1.0, alpha=1.0, period=5.0):
    # Quasi-periodic kernel (periodic * SE decay)
    return (sigma_f**2 *
            np.exp(-0.5*(x-xp)**2/ell**2) *
            np.exp(-2*np.sin(np.pi*np.abs(x-xp)/period)**2/alpha**2))

def sample_gp(kernel, x, n_samples=5, noise=1e-6):
    # Sample n functions from GP(0, k)
    K  = kernel(*np.meshgrid(x, x))
    K += noise * np.eye(len(x))
    L  = cholesky(K, lower=True)
    return L @ np.random.randn(len(x), n_samples)

x_plot = np.linspace(0, 10, 200)

kernels = [
    (k_se,       {"sigma_f":1,"ell":1.0},          "Squared Exponential
(ℓ=1.0)",   "#3B8BD4"),
    (k_matern32, {"sigma_f":1,"ell":1.0},          "Matérn 3/2
(ℓ=1.0)",           "#1D9E75"),
    (k_periodic, {"sigma_f":1,"ell":1.0,"period":5},"Periodic
(P=5, ℓ=1.0)",       "#D85A30"),
    (k_qp,       {"sigma_f":1,"ell":3.0,"alpha":1.0,"period":5},"Quasi-Periodic
(P=5, decay ℓ=3)", "#7F77DD"),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for col, (kern, kw, title, col_c) in enumerate(kernels):
    # Kernel function (as function of r = |x-x'|)
    r = np.linspace(0, 6, 200)
    k_vals = kern(0, r, **kw)
    axes[0, col].plot(r, k_vals, color=col_c, lw=2.5)
    axes[0, col].set_xlabel("Separation |x-x'|")
    axes[0, col].set_ylabel("k(x, x')")
    axes[0, col].set_title(f"{title}
kernel function")
    axes[0, col].set_ylim(-0.05, 1.15)

    # Sample functions from GP prior
    np.random.seed(col)
    kern_fn = lambda x, xp, k=kern, kw=kw: k(x, xp, **kw)
    samples = sample_gp(kern_fn, x_plot)
    for s in range(samples.shape[1]):
        axes[1, col].plot(x_plot, samples[:, s], color=col_c, alpha=0.5, lw=1.5)
    axes[1, col].set_xlabel("x")
    axes[1, col].set_ylabel("f(x)")
    axes[1, col].set_title("5 sample functions from GP prior")

plt.suptitle("GP Kernels and Corresponding Sample Functions", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## 7.2 GP Regression from Scratch

Given noisy observations y = f(x) + ε, ε ~ N(0, σ_n²):

$$\mu_* = K_{*n}(K_{nn} + \sigma_n^2 I)^{-1}\mathbf{y}$$
$$\Sigma_* = K_{**} - K_{*n}(K_{nn} + \sigma_n^2 I)^{-1}K_{n*}$$


In [ ]:
def gp_predict(x_train, y_train, x_pred, kernel, sigma_n):
    # GP regression: exact predictive mean and variance
    K_nn  = kernel(*np.meshgrid(x_train, x_train)) + sigma_n**2 * np.eye(len(x_train))
    K_sn  = kernel(*np.meshgrid(x_train, x_pred)).T     # (n_pred, n_train)
    K_ss  = np.diag(kernel(x_pred, x_pred))

    L = cholesky(K_nn, lower=True)
    alpha = solve_triangular(L.T, solve_triangular(L, y_train, lower=True))
    mu    = K_sn @ alpha

    V     = solve_triangular(L, K_sn.T, lower=True)
    var   = K_ss - np.sum(V**2, axis=0)
    return mu, np.sqrt(np.clip(var, 0, None))

# Fit a GP to stellar light curve snippet
np.random.seed(3)
n_obs     = 15
x_obs     = np.sort(np.random.uniform(0, 10, n_obs))
y_true_fn = lambda x: 1.5*np.sin(x) + 0.4*np.cos(2.5*x)
y_obs     = y_true_fn(x_obs) + 0.3*np.random.randn(n_obs)
sigma_n   = 0.3

x_pred = np.linspace(-0.5, 10.5, 300)
kernel_se = lambda x, xp: k_se(x, xp, sigma_f=1.5, ell=1.5)

mu_pred, std_pred = gp_predict(x_obs, y_obs, x_pred, kernel_se, sigma_n)

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(x_pred, mu_pred-2*std_pred, mu_pred+2*std_pred,
                color="#3B8BD4", alpha=0.15, label="±2σ posterior")
ax.fill_between(x_pred, mu_pred-std_pred,   mu_pred+std_pred,
                color="#3B8BD4", alpha=0.30, label="±1σ posterior")
ax.plot(x_pred, mu_pred, "#3B8BD4", lw=2.5, label="Posterior mean")
ax.plot(x_pred, y_true_fn(x_pred), "#1D9E75", lw=2, ls="--", alpha=0.7, label="True function")
ax.scatter(x_obs, y_obs, color="black", s=50, zorder=5, label="Observations")
ax.set_xlabel("Time (days)"); ax.set_ylabel("Flux deviation")
ax.set_title("GP Regression — Stellar Light Curve Interpolation
SE kernel: σf=1.5, ℓ=1.5")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

# Show how uncertainty grows between observations
gap_region = (x_pred > 4) & (x_pred < 6)
print(f"Mean posterior σ in gap region (4–6 days): {std_pred[gap_region].mean():.3f}")
print(f"Mean posterior σ near observations:         {std_pred[~gap_region].mean():.3f}")


## 7.3 Hyperparameter Optimisation via Marginal Likelihood

The GP hyperparameters (σf, ℓ, σn) can be optimised by maximising the **log marginal likelihood**:

$$\ln P(\mathbf{y} \mid X, \phi) = -\frac{1}{2}\mathbf{y}^T C^{-1}\mathbf{y} - \frac{1}{2}\ln|C| - \frac{n}{2}\ln 2\pi$$

This balances fit quality against model complexity (Occam factor = -½ ln|C|).


In [ ]:
from scipy.optimize import minimize

def log_marginal_likelihood(log_params, x_train, y_train):
    # Log marginal likelihood for SE kernel GP
    sigma_f, ell, sigma_n = np.exp(log_params)
    K = k_se(*np.meshgrid(x_train, x_train), sigma_f=sigma_f, ell=ell)
    C = K + sigma_n**2 * np.eye(len(x_train))
    try:
        L    = cholesky(C, lower=True)
        alpha = solve_triangular(L.T, solve_triangular(L, y_train, lower=True))
        lml  = -0.5 * y_train @ alpha - np.sum(np.log(np.diag(L))) - 0.5*len(y_train)*np.log(2*np.pi)
        return -lml   # negative for minimisation
    except Exception:
        return 1e10

# Optimise hyperparameters
res = minimize(log_marginal_likelihood, x0=np.log([1.0, 1.0, 0.3]),
               args=(x_obs, y_obs), method="L-BFGS-B",
               bounds=[(-3,3),(-3,3),(-5,2)])
sigma_f_opt, ell_opt, sigma_n_opt = np.exp(res.x)
print(f"Optimised hyperparameters:")
print(f"  σf = {sigma_f_opt:.3f}  (prior: 1.5)")
print(f"  ℓ  = {ell_opt:.3f}  (prior: 1.5)")
print(f"  σn = {sigma_n_opt:.3f}  (prior: 0.3)")

# Compare: manual vs optimised hyperparameters
kernel_opt = lambda x, xp: k_se(x, xp, sigma_f=sigma_f_opt, ell=ell_opt)
mu_opt, std_opt = gp_predict(x_obs, y_obs, x_pred, kernel_opt, sigma_n_opt)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (mu, std, title, col) in zip(axes, [
    (mu_pred, std_pred, "Manual hyperparams (σf=1.5, ℓ=1.5, σn=0.3)", "#3B8BD4"),
    (mu_opt,  std_opt,  f"Optimised (σf={sigma_f_opt:.2f}, ℓ={ell_opt:.2f}, σn={sigma_n_opt:.3f})", "#D85A30"),
]):
    ax.fill_between(x_pred, mu-std, mu+std, color=col, alpha=0.3)
    ax.plot(x_pred, mu, color=col, lw=2.5, label="Posterior mean")
    ax.plot(x_pred, y_true_fn(x_pred), "#1D9E75", lw=1.5, ls="--", alpha=0.7, label="Truth")
    ax.scatter(x_obs, y_obs, color="black", s=40, zorder=5)
    ax.set_title(title); ax.set_xlabel("Time"); ax.legend(fontsize=9, frameon=False)

plt.tight_layout()
plt.show()


## 7.4 Transit + GP Joint Fit (Concept)

In exoplanet analysis, stellar activity creates correlated noise that can
bias transit depth measurements. The joint model is:

$$y_i = F_{\rm transit}(t_i;\, \theta) + f_{\rm GP}(t_i;\, \phi) + \epsilon_i$$

The GP models the stellar systematics; the transit model extracts the planet signal.
`celerite2` implements this at O(n) cost.


In [ ]:
try:
    import celerite2
    from celerite2 import terms

    # Demonstrate celerite2 GP on a simple stellar variability example
    np.random.seed(8)
    t = np.sort(np.random.uniform(0, 20, 100))
    # Simulate stellar rotation at P=8 days + white noise
    y_star = 0.5*np.sin(2*np.pi*t/8) + 0.2*np.cos(4*np.pi*t/8)
    yerr   = 0.1*np.ones_like(t)
    y_obs_star = y_star + yerr*np.random.randn(len(t))

    # Build a celerite2 GP with rotation term
    kernel = terms.RotationTerm(sigma=0.5, period=8.0, Q0=0.5, dQ=0.5, f=0.5)
    gp = celerite2.GaussianProcess(kernel, mean=0.0)
    gp.compute(t, yerr=yerr)

    # Predict on a fine grid
    t_pred = np.linspace(0, 20, 500)
    mu_c, var_c = gp.predict(y_obs_star, t=t_pred, return_var=True)
    std_c = np.sqrt(var_c)

    print(f"celerite2 GP log-likelihood: {gp.log_likelihood(y_obs_star):.2f}")

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.fill_between(t_pred, mu_c-std_c, mu_c+std_c, color="#7F77DD", alpha=0.35)
    ax.plot(t_pred, mu_c, "#7F77DD", lw=2.5, label="celerite2 GP mean")
    ax.plot(t_pred, 0.5*np.sin(2*np.pi*t_pred/8)+0.2*np.cos(4*np.pi*t_pred/8),
            "#1D9E75", lw=2, ls="--", label="True stellar signal")
    ax.errorbar(t, y_obs_star, yerr=yerr, fmt="o", color="#888780", ms=4, elinewidth=0.8,
                label="Observations")
    ax.set_xlabel("Time (days)"); ax.set_ylabel("Flux deviation")
    ax.set_title("celerite2 RotationTerm GP — Stellar Variability Model
P=8 days (solar-like rotation)")
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()

except ImportError:
    print("celerite2 not installed. Run: pip install celerite2")
    print("The celerite2 RotationTerm is used in the capstone project for stellar noise modelling.")


In [ ]:
print("Chapter 7 complete!")
print()
print("Key kernels for astrophysics:")
print("  - SE (RBF):        smooth slow signals (spectral continua, slow variability)")
print("  - Matérn 3/2:      less smooth (stellar flares, transients)")
print("  - Quasi-periodic:  stellar rotation with decay (activity cycle)")
print("  - Matérn 1/2:      very rough / uncorrelated — rarely useful")
print()
print("The celerite trick: separable kernels → O(n) GP → feasible for Kepler/TESS data")
